Ejemplo de Interfaz para Predicción de Churn


In [1]:
!pip install gradio

In [2]:
import pandas as pd

# Sample dataset
df = pd.DataFrame({
    "edad": [25,45,34,52,23,44,31,60],
    "ingresos": [2000,5000,3200,6000,1800,5200,3100,7000],
    "uso": [10,30,20,40,5,25,15,50],
    "soporte": [1,5,2,3,0,4,1,6],
    "satisfaccion": [4,2,3,5,5,2,4,1],
    "churn": [0,1,0,1,0,1,0,1]
})
df.to_csv("churn_dataset.csv", index=False)
df.head()

,edad,ingresos,uso,soporte,satisfaccion,churn
0,25,2000,10,1,4,0
1,45,5000,30,5,2,1
2,34,3200,20,2,3,0
3,52,6000,40,3,5,1
4,23,1800,5,0,5,0


In [13]:
import pandas as pd
import numpy as np

np.random.seed(42)  # reproducible

N = 200

# --- Feature Generation ---
edad = np.random.randint(18, 70, N)

ingresos = np.random.normal(3500, 1200, N).clip(1200, 9000)

uso_mensual = np.random.randint(1, 60, N)

llamadas_soporte = np.random.poisson(2, N).clip(0, 10)

satisfaccion = np.random.randint(1, 6, N)   # 1–5 rating

# --- Churn probability model ---
# Higher probability with:
# - low satisfaction
# - high support calls
# - low usage
# - low income

prob_churn = (
    (6 - satisfaccion) * 0.12 +
    llamadas_soporte * 0.07 +
    (1 / (uso_mensual + 1)) * 0.8 +
    (3000 - ingresos).clip(0,3000) / 3000 * 0.2 +
    np.random.normal(0, 0.05, N)
)

# Clip probabilities between 0 and 1
prob_churn = prob_churn.clip(0, 1)

# Generate binary churn based on probability
churn = np.random.binomial(1, prob_churn)

# --- Create DataFrame ---
df = pd.DataFrame({
    "edad": edad,
    "ingresos": ingresos.astype(int),
    "uso": uso_mensual,
    "soporte": llamadas_soporte,
    "satisfaccion": satisfaccion,
    "churn": churn
})

df.to_csv("churn_dataset.csv", index=False)
df.head(), df.shape


(   edad  ingresos  uso  soporte  satisfaccion  churn
 0    56      3652   11        1             1      1
 1    69      2000   29        2             5      1
 2    46      5834   56        0             1      1
 3    32      3315   36        4             1      1
 4    60      2411   25        2             1      1,
 (200, 6))

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("churn_dataset.csv")
X = df.drop("churn", axis=1)
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and fit the scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the model with scaled data
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)

model.score(X_test_scaled, y_test)

0.7

In [20]:
from sklearn.metrics import confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

# Make predictions on test set
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)


In [16]:
def plot_confusion_matrix(cm):
    fig, ax = plt.subplots(figsize=(4,4))
    ax.imshow(cm, cmap="Blues")
    ax.set_title("Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    # Labels
    ax.set_xticks(np.arange(2))
    ax.set_yticks(np.arange(2))
    ax.set_xticklabels(["No Churn","Churn"])
    ax.set_yticklabels(["No Churn","Churn"])

    # Write values in cells
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")

    return fig


def plot_roc_curve(fpr, tpr, roc_auc):
    fig, ax = plt.subplots(figsize=(4,4))
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
    ax.plot([0,1], [0,1], linestyle="--", color="gray")
    ax.set_title("ROC Curve")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend()
    return fig


In [17]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test_scaled)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.47      0.48        17
           1       0.62      0.65      0.64        23

    accuracy                           0.57        40
   macro avg       0.56      0.56      0.56        40
weighted avg       0.57      0.57      0.57        40


Confusion Matrix:
[[ 8  9]
 [ 8 15]]


In [18]:
import gradio as gr

# Get min/max for each feature
min_edad, max_edad = int(df["edad"].min()), int(df["edad"].max())
min_ing, max_ing = int(df["ingresos"].min()), int(df["ingresos"].max())
min_uso, max_uso = int(df["uso"].min()), int(df["uso"].max())
min_sop, max_sop = int(df["soporte"].min()), int(df["soporte"].max())
min_sat, max_sat = int(df["satisfaccion"].min()), int(df["satisfaccion"].max())

def predict_churn(edad, ingresos, uso, soporte, satisfaccion):
    data = [[edad, ingresos, uso, soporte, satisfaccion]]
    # Scale the input data using the *fitted* scaler
    scaled_data = scaler.transform(data)
    result = model.predict(scaled_data)[0]
    prob = model.predict_proba(scaled_data)[0][1]
    return f"Churn: {result} – Probabilidad: {prob:.2f}"

# ----- Nice Looking SOFT Theme -----
theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="blue",
    neutral_hue="slate"
)

interface = gr.Interface(
    fn=predict_churn,
    inputs=[
        gr.Slider(min_edad, max_edad, step=1, label="Edad"),
        gr.Slider(min_ing, max_ing, step=50, label="Ingresos Mensuales"),
        gr.Slider(min_uso, max_uso, step=1, label="Uso Mensual"),
        gr.Slider(min_sop, max_sop, step=1, label="Llamadas a Soporte"),
        gr.Slider(min_sat, max_sat, step=1, label="Satisfacción"),
    ],
    outputs="text",
    theme=theme,
    title="Predicción de Churn con KNN",
    description="Modifica los valores en los sliders para ver cómo cambia la predicción del modelo."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9edc5ab1423f763fdf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
